# 🎹 자동 채보 — 내 연주를 MIDI로

피아노 연주 오디오 파일을 넣으면, AI가 음표를 인식하여 **MIDI (악보 데이터)**로 변환합니다.
이 MIDI를 나중에 AI 작곡의 입력으로 사용합니다.

**도구**: [Basic Pitch](https://basicpitch.spotify.com/) — Spotify가 만든 오픈소스 채보 AI

▶ 맨 먼저 아래 셀을 실행해주세요. 설치에 2~3분 정도 걸립니다.

In [ ]:
%%capture
# 필요한 패키지 설치
!apt-get -qq install -y fluidsynth > /dev/null 2>&1
!pip install -q basic-pitch librosa soundfile midi2audio matplotlib pretty_midi

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')

---
## 1. 내 연주 파일 올리기

▶ 본인 피아노 연주 파일을 업로드하세요. (mp3 또는 wav)

스마트폰으로 녹음한 파일도 괜찮습니다. 30초~1분 정도면 충분합니다.

녹음 파일이 없으면 이 셀은 건너뛰고, 다음 셀에서 예시 파일을 사용하세요.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
input_audio = list(uploaded.keys())[0]
print(f'✅ 파일이 업로드되었습니다: {input_audio}')

---
## 2. (선택) 예시 파일 사용

▶ 녹음 파일이 없으면 이 셀을 실행하세요. 예시 피아노 연주를 다운로드합니다.

In [ ]:
import urllib.request
import IPython.display as ipd

# 예시 파일 다운로드
REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'

examples = {
    'piano_chopin.wav': '쇼팽 녹턴 발췌 (30초)',
    'piano_jazz.wav': '재즈 피아노 즉흥 (30초)',
    'piano_simple.wav': '간단한 스케일 (15초)',
}

for fname, desc in examples.items():
    if not os.path.exists(fname):
        try:
            urllib.request.urlretrieve(f'{REPO_URL}/{fname}', fname)
            print(f'  ✅ {desc}: {fname}')
        except:
            print(f'  ⚠️ {fname} 다운로드 실패 (assets 폴더에 파일이 아직 없을 수 있습니다)')

# ← 다른 파일로 바꿔보세요
input_audio = 'piano_chopin.wav'

print(f'\n🎵 선택된 파일: {input_audio}')
if os.path.exists(input_audio):
    ipd.display(ipd.Audio(input_audio))
else:
    print('⚠️ 파일이 없습니다. 위 셀에서 직접 업로드해주세요.')

---
## 3. AI로 음표 인식하기

Basic Pitch는 오디오의 파형을 분석해서 어떤 음이 언제, 얼마나 세게 연주되었는지를
자동으로 알아냅니다. 사람이 귀로 듣고 악보를 쓰는 것과 같은 일을 AI가 하는 것입니다.

▶ 아래 셀을 실행하세요.

In [ ]:
from basic_pitch.inference import predict

print('🔄 AI가 음표를 분석하고 있습니다...')
model_output, midi_data, note_events = predict(input_audio)

# MIDI 파일 저장
output_midi = 'output.mid'
midi_data.write(output_midi)

print(f'✅ 변환 완료! {len(note_events)}개의 음표를 찾았습니다.')
print(f'💾 MIDI 파일 저장됨: {output_midi}')

---
## 4. 결과 확인 — 피아노롤

아래는 AI가 인식한 음표를 시각화한 **피아노롤**입니다.
- 가로축: 시간 (초)
- 세로축: 음높이 (높을수록 고음)
- 색상: 세기 (진할수록 강하게 연주)

아래에서 변환된 MIDI를 다시 소리로 들어볼 수도 있습니다.
원본 연주와 비교해보세요 — 어떤 차이가 있나요?

In [ ]:
import pretty_midi
import matplotlib.pyplot as plt
import numpy as np
from midi2audio import FluidSynth
import IPython.display as ipd

# 피아노롤 시각화
pm = pretty_midi.PrettyMIDI(output_midi)

fig, ax = plt.subplots(figsize=(14, 5))

for instrument in pm.instruments:
    for note in instrument.notes:
        ax.barh(
            note.pitch,
            note.end - note.start,
            left=note.start,
            height=0.8,
            color=plt.cm.viridis(note.velocity / 127),
            alpha=0.8
        )

# y축 범위를 실제 음역대에 맞춤
all_pitches = [n.pitch for inst in pm.instruments for n in inst.notes]
if all_pitches:
    ax.set_ylim(min(all_pitches) - 2, max(all_pitches) + 2)

ax.set_xlabel('시간 (초)', fontsize=12)
ax.set_ylabel('음높이 (MIDI 번호)', fontsize=12)
ax.set_title('🎹 AI가 인식한 피아노롤', fontsize=14)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# MIDI → 오디오 변환
print('\n🔊 MIDI를 소리로 변환 중...')
midi_audio = 'output_midi_audio.wav'
try:
    fs = FluidSynth()
    fs.midi_to_audio(output_midi, midi_audio)
    print('\n🎵 변환된 MIDI 소리:')
    ipd.display(ipd.Audio(midi_audio))
except:
    print('⚠️ MIDI 오디오 변환에 실패했습니다. FluidSynth 사운드폰트가 없을 수 있습니다.')

if os.path.exists(input_audio):
    print('\n🎵 원본 연주:')
    ipd.display(ipd.Audio(input_audio))

---
## 5. MIDI 파일 다운로드

▶ 아래 셀을 실행하면 MIDI 파일이 다운로드됩니다.

💡 이 MIDI를 **'내 멜로디 + AI' 노트북**에서 사용합니다. 꼭 저장해두세요!

In [ ]:
from google.colab import files
files.download(output_midi)
print('✅ 다운로드 완료!')